<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-10/AI_Modul_10.2_Praktikum_Representasi_Citra_Digital.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 10.2: Praktikum Representasi Citra Digital dan Matriks Piksel
**Mata Kuliah:** Visi Komputer dan Kecerdasan Buatan Terapan  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Memahami struktur representasi matriks citra digital 2D (Grayscale) dan tensor 3D (RGB/BGR) menggunakan NumPy dan OpenCV.
2. Mampu mengekstraksi dan memanipulasi Region of Interest (ROI) pada komoditas kelapa sawit dengan mekanisme deep copy yang aman.
3. Mampu membuktikan disparitas perilaku aritmatika piksel `uint8` antara NumPy modulo vs OpenCV bersaturasi.
4. Mampu merancang fungsi Lookup Table (LUT) untuk transformasi kontras linier dan koreksi gamma non-linier guna mencerahkan bayangan kanopi perkebunan.
5. Mampu mendeteksi fenomena kuantisasi *false contouring* akibat reduksi kedalaman bit citra.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'OpenCV Version : {cv2.__version__}')
print(f'NumPy Version  : {np.__version__}')


## 1. Pembangkitan Matriks Citra Sintetis & Karakteristik Tensor
Kita membangkitkan citra sintetis brondolan kelapa sawit (*oil palm fruitlet*) dengan gradasi kematangan warna (merah-oranye mesokarp) pada latar belakang gelap.

In [ ]:
width, height = 300, 300
palm_fruit = np.ones((height, width, 3), dtype=np.uint8) * 35  # Background gelap

Y, X = np.meshgrid(np.arange(height), np.arange(width), indexing='ij')
cy, cx = height // 2, width // 2

# Geometri elips brondolan sawit
a, b = 100.0, 70.0
mask = ((X - cx) / b)**2 + ((Y - cy) / a)**2 <= 1.0

y_norm = np.clip((Y - (cy - a)) / (2 * a), 0.0, 1.0)
# Saluran Merah (R): Tinggi di badan buah
palm_fruit[mask, 0] = np.clip(180 + 75 * (1.0 - y_norm[mask]), 0, 255).astype(np.uint8)
# Saluran Hijau (G): Gradasi kematangan menuju dasar
palm_fruit[mask, 1] = np.clip(35 + 85 * (1.0 - y_norm[mask]**2), 0, 255).astype(np.uint8)
# Saluran Biru (B): Rendah khas buah sawit matang
palm_fruit[mask, 2] = np.clip(15 + 15 * y_norm[mask], 0, 255).astype(np.uint8)

print('=== PROFIL TENSOR CITRA SAWIT ===')
print(f'Dimensi Array (H, W, C) : {palm_fruit.shape}')
print(f'Tipe Data Elemen        : {palm_fruit.dtype}')
print(f'Ukuran Memori Total     : {palm_fruit.nbytes:,} Bytes ({palm_fruit.nbytes / 1024:.2f} KB)')
print(f'Rentang Nilai Intensitas: Min = {palm_fruit.min()}, Max = {palm_fruit.max()}')


## 2. Ekstraksi Region of Interest (ROI) & Pembuktian View vs Copy
Dalam NumPy, pengirisan `img[y1:y2, x1:x2]` membentuk *view* (referensi memori bersama). Memodifikasi *view* akan merusak citra asal, sehingga wajib menggunakan `.copy()` jika data mentah harus dipertahankan.

In [ ]:
# Ekstraksi ROI Bagian Inti Buah Sawit Menggunakan .copy()
ymin, ymax = 100, 200
xmin, xmax = 100, 200
roi_safe = palm_fruit[ymin:ymax, xmin:xmax].copy()

# Ekstraksi ROI View (Referensi Berbahaya)
fruit_demo = palm_fruit.copy()
roi_view = fruit_demo[ymin:ymax, xmin:xmax]
# Mengubah ROI view menjadi putih pekat
roi_view[:, :] = [255, 255, 255]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(palm_fruit)
axes[0].set_title('(A) Citra Asli Utuh')
axes[0].axis('off')

axes[1].imshow(roi_safe)
axes[1].set_title(f'(B) ROI Terisolasi (.copy()) {roi_safe.shape}')
axes[1].axis('off')

axes[2].imshow(fruit_demo)
axes[2].set_title('(C) Dampak Mutasi Memori pada View')
axes[2].axis('off')
plt.tight_layout()
plt.show()


## 3. Bahaya Aritmatika Piksel uint8: Modulo vs Saturasi
Tipe data `uint8` berada pada rentang $[0, 255]$. Penjumlahan murni NumPy bersifat modulo (meluap kembali ke 0), sedangkan OpenCV menggunakan fungsi jenuh bersaturasi `cv2.add`.

In [ ]:
# Uji Penambahan Kecerahan +60 pada Citra Sawit
brightness_bias = 60

# Alternatif A: NumPy Add (Modulo 256)
img_numpy_overflow = palm_fruit + brightness_bias

# Alternatif B: OpenCV Add (Saturasi [0, 255])
bias_array = np.full(palm_fruit.shape, brightness_bias, dtype=np.uint8)
img_opencv_saturated = cv2.add(palm_fruit, bias_array)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(palm_fruit)
axes[0].set_title('(A) Citra Asli (Rerata = {:.1f})'.format(palm_fruit.mean()))
axes[0].axis('off')

axes[1].imshow(img_numpy_overflow)
axes[1].set_title('(B) NumPy Add (Cacat Overflow Modulo)')
axes[1].axis('off')

axes[2].imshow(img_opencv_saturated)
axes[2].set_title('(C) OpenCV Add (Saturasi Sempurna)')
axes[2].axis('off')
plt.tight_layout()
plt.show()


## 4. Transformasi Intensitas Titik: Koreksi Gamma Adaptif via Lookup Table (LUT)
Fungsi non-linier $s = r^{1/\gamma}$ mencerahkan area gelap berbayang (ketika $\gamma > 1.0$) atau meredam area silau (ketika $\gamma < 1.0$). Penggunaan tabel pemetaan `cv2.LUT` menjamin eksekusi $\mathcal{O}(1)$ tanpa overhead perhitungan eksponensial per piksel.

In [ ]:
def gamma_correction(image, gamma=1.0):
    inv_gamma = 1.0 / gamma
    lut = np.array([
        np.clip(((i / 255.0) ** inv_gamma) * 255.0, 0, 255)
        for i in range(256)
    ], dtype=np.uint8)
    return cv2.LUT(image, lut)

# Uji Tiga Nilai Gamma
fruit_gamma_05 = gamma_correction(palm_fruit, gamma=0.5)  # Meredam silau / menggelapkan
fruit_gamma_10 = gamma_correction(palm_fruit, gamma=1.0)  # Identitas
fruit_gamma_22 = gamma_correction(palm_fruit, gamma=2.2)  # Mencerahkan bayangan gelap

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(fruit_gamma_05)
axes[0].set_title('(A) Gamma = 0.5 (Peredaman Highlights)')
axes[0].axis('off')

axes[1].imshow(fruit_gamma_10)
axes[1].set_title('(B) Gamma = 1.0 (Identitas Asli)')
axes[1].axis('off')

axes[2].imshow(fruit_gamma_22)
axes[2].set_title('(C) Gamma = 2.2 (Pencerahan Detail Bayangan)')
axes[2].axis('off')
plt.tight_layout()
plt.show()


## 5. Simulasi Kuantisasi Intensitas & Efek False Contouring
Ketika kedalaman bit direduksi dari 8-bit (256 tingkat) menjadi 4-bit (16 tingkat) atau 2-bit (4 tingkat), gradasi warna halus akan pecah menjadi garis batas kontur semu (*false contouring / banding*).

In [ ]:
def quantize_bits(image, bits):
    levels = 2 ** bits
    step = 256 // levels
    return ((image // step) * step).astype(np.uint8)

fruit_8bit = palm_fruit
fruit_4bit = quantize_bits(palm_fruit, bits=4)  # 16 Level
fruit_2bit = quantize_bits(palm_fruit, bits=2)  # 4 Level

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(fruit_8bit)
axes[0].set_title('(A) Kedalaman 8-bit (256 Tingkat Warna)')
axes[0].axis('off')

axes[1].imshow(fruit_4bit)
axes[1].set_title('(B) Kedalaman 4-bit (16 Tingkat: Muncul Banding)')
axes[1].axis('off')

axes[2].imshow(fruit_2bit)
axes[2].set_title('(C) Kedalaman 2-bit (4 Tingkat: Kontur Palsu Parah)')
axes[2].axis('off')
plt.tight_layout()
plt.show()
